# MDS-SV Relationship Analysis

This notebook tests whether digital movement density (MDS) predicts sales velocity (SV).

**Analyses:**
1. Correlation analysis (Pearson and Spearman)
2. OLS regression with fixed effects
3. Lead-lag analysis (does MDS predict future SV?)
4. Robustness tests
5. Visualizations

**Input:** Aligned MDS/SV data from `02_build_scores.ipynb`

**Output:** Statistical results, plots, and summary report

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path

# Import analysis modules
from src.analysis.models import run_correlations_and_models, MDSSVAnalysis
from src.analysis.plots import make_plots

In [ ]:
# Load aligned data
aligned_path = Path('../data/processed/aligned_mds_sv.csv')
aligned_df = pd.read_csv(aligned_path)

print(f"Loaded {len(aligned_df)} observations")
print(f"Shows: {aligned_df['show_name'].nunique()}")
print(f"Cohorts: {aligned_df['cohort'].unique().tolist()}")

# Basic statistics
print("\nMDS Statistics:")
print(aligned_df['mds'].describe())
print("\nSV Statistics:")
print(aligned_df['sv'].describe())

## 1. Correlation Analysis

In [ ]:
# Run correlation analysis
analysis = MDSSVAnalysis(aligned_df)
corr_results = analysis.run_correlations()

print("=" * 60)
print("CORRELATION ANALYSIS")
print("=" * 60)
print(f"Sample size: {corr_results['n_obs']} observations")
print(f"Number of shows: {corr_results['n_shows']}")
print(f"\nPearson correlation: r = {corr_results['pearson_r']:.4f}")
print(f"  p-value: {corr_results['pearson_p']:.6f}")
print(f"\nSpearman correlation: ρ = {corr_results['spearman_r']:.4f}")
print(f"  p-value: {corr_results['spearman_p']:.6f}")
print("=" * 60)

# Interpretation
if corr_results['pearson_p'] < 0.05:
    print("\n✓ Statistically significant correlation detected (p < 0.05)")
else:
    print("\n✗ No statistically significant correlation detected (p >= 0.05)")

## 2. OLS Regression with Fixed Effects

In [ ]:
# Run OLS regression
ols_results = analysis.run_ols_regression(include_fixed_effects=True)

if 'error' not in ols_results:
    print("=" * 60)
    print("OLS REGRESSION: SV ~ MDS + Show FE + Week FE")
    print("=" * 60)
    print(f"N = {ols_results['n_obs']}")
    print(f"R-squared: {ols_results['r_squared']:.4f}")
    print(f"Adjusted R-squared: {ols_results['r_squared_adj']:.4f}")
    print(f"Residual Std Error: {ols_results['residual_std_error']:.4f}")
    print("\nMDS Coefficient:")
    
    mds_coef = ols_results['coefficients']['mds']
    print(f"  Estimate: {mds_coef['estimate']:.4f}")
    print(f"  Std Error: {mds_coef['std_error']:.4f}")
    print(f"  t-statistic: {mds_coef['t_stat']:.4f}")
    print(f"  p-value: {mds_coef['p_value']:.6f}")
    print(f"  95% CI: [{mds_coef['ci_lower']:.4f}, {mds_coef['ci_upper']:.4f}]")
    print("=" * 60)
    
    if mds_coef['p_value'] < 0.05:
        print("\n✓ MDS is a significant predictor of SV (p < 0.05)")
    else:
        print("\n✗ MDS is not a significant predictor (p >= 0.05)")
else:
    print(f"Error in OLS regression: {ols_results['error']}")

## 3. Lead-Lag Analysis

In [ ]:
# Test whether MDS predicts future SV
lead_lag_results = analysis.run_lead_lag_analysis(max_lag=4)

print("=" * 60)
print("LEAD-LAG ANALYSIS: Does MDS(t) predict SV(t+L)?")
print("=" * 60)
print(lead_lag_results.to_string(index=False))
print("=" * 60)

# Find strongest lag
strongest_lag = lead_lag_results.loc[lead_lag_results['pearson_r'].abs().idxmax()]
print(f"\nStrongest correlation at lag {int(strongest_lag['lag'])} weeks:")
print(f"  r = {strongest_lag['pearson_r']:.4f}, p = {strongest_lag['p_value']:.6f}")

## 4. Robustness Tests

In [ ]:
# Run robustness checks
robustness_results = analysis.run_robustness_tests()

print("=" * 60)
print("ROBUSTNESS TESTS")
print("=" * 60)

for test_name, results in robustness_results.items():
    print(f"\n{test_name.replace('_', ' ').title()}:")
    print(f"  N = {results['n_obs']}")
    print(f"  Pearson r = {results['pearson_r']:.4f}")
    print(f"  p-value = {results['pearson_p']:.6f}")

print("=" * 60)

## 5. Generate Visualizations

In [ ]:
# Generate all plots
output_dir = Path('../outputs')

make_plots(
    aligned_df=aligned_df,
    lead_lag_df=lead_lag_results,
    correlation_stats=corr_results,
    output_dir=output_dir,
)

print(f"\nAll visualizations saved to {output_dir}/")

## 6. Generate Summary Report

In [ ]:
# Generate comprehensive summary
summary_report = analysis.generate_summary_report()

print(summary_report)

# Save to file
report_path = Path('../outputs/analysis_summary_report.txt')
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    f.write(summary_report)

print(f"\nSummary report saved to {report_path}")

## 7. Data Quality Assessment

In [ ]:
# Assess data completeness
print("=" * 60)
print("DATA QUALITY ASSESSMENT")
print("=" * 60)

# Check missing data by show
print("\nMissing Data by Show:")
missing_summary = aligned_df.groupby('show_name').agg({
    'mds': lambda x: (x.isna().sum() / len(x) * 100),
    'sv': lambda x: (x.isna().sum() / len(x) * 100),
})
missing_summary.columns = ['MDS missing %', 'SV missing %']
print(missing_summary.round(1))

# Overall completeness
total_mds_missing = aligned_df['mds'].isna().sum()
total_sv_missing = aligned_df['sv'].isna().sum()
total_rows = len(aligned_df)

print(f"\nOverall Completeness:")
print(f"  MDS: {(1 - total_mds_missing/total_rows)*100:.1f}% complete")
print(f"  SV: {(1 - total_sv_missing/total_rows)*100:.1f}% complete")
print(f"  Both: {(aligned_df['mds'].notna() & aligned_df['sv'].notna()).sum()} observations")

print("=" * 60)

## Summary

This notebook completed the full analysis pipeline:

1. **Correlation Analysis**: Tested association between MDS and SV
2. **OLS Regression**: Quantified MDS effect controlling for show and time fixed effects
3. **Lead-Lag**: Tested whether MDS predicts future SV (predictive power)
4. **Robustness**: Validated results across different subsets and specifications
5. **Visualizations**: Generated publication-quality plots

**Key Findings:** See summary report above

**Limitations:**
- Public data only (limited Instagram/TikTok access)
- Sales proxies not actual ticket sales
- Correlational evidence only (not causal)
- Small sample size for some shows

**Next Steps:**
- Manual verification of TODO items in config/shows.yaml
- Fill data gaps where possible
- Consider additional robustness checks